# Chapter 5-4. 백테스팅 심화 & 워크포워드 분석 (Walk-Forward Analysis)

> **KDT AI 퀀트 · 백테스트/성과검증 파트 심화 실습**

앞선 실습에서는 삼성전자 데이터로 단순 평균회귀 전략을 `for` 루프로 백테스팅했습니다.
이번 챕터에서는 조금 더 실전적인 **추세 필터 + RSI 전략**으로 예제를 바꾸고,
이를 **재사용 가능한 함수**로 정리한 뒤 **워크포워드 방법론**까지 다룹니다.

### 📈 이번 챕터의 예제 전략: "추세 위에서 눌림목 사기"

| 항목 | 규칙 |
|------|------|
| **레짐 필터(매수 가능 구간)** | 종가가 **200일 이동평균선 위**에 있을 때만 매수 허용 (상승 추세 확인) |
| **진입(Buy)** | RSI가 **45를 상향 돌파**할 때 (전일 RSI < 45 → 당일 RSI ≥ 45) |
| **청산(Sell)** | RSI가 **80에 도달**할 때 전량 매도 |

> 아이디어: **큰 추세는 위(200일선 위)** 인데, 단기적으로 눌린(RSI 조정) 뒤 반등이 시작되는
> 지점을 노려 진입하고, 과열(RSI 80)에서 차익을 실현하는 전략입니다.
>
> ⚠️ **왜 진입 기준이 30이 아니라 45인가?** 처음에는 "과매도(RSI 30) 돌파"로 설계했지만,
> RSI가 30까지 내려갈 만큼 급락하면 가격도 대개 200일선 아래로 떨어져서
> **'RSI<30' 과 'price>MA200' 두 조건이 거의 동시에 성립하지 않습니다**(뒤 진단 셀 참고).
> 상승 추세의 **얕은 눌림목**을 잡으려면 RSI 45~50 수준의 반등을 노리는 것이 정석입니다.

| 단계 | 내용 |
|------|------|
| **1. 리팩토링** | 전략·성과지표를 **재사용 가능한 함수**로 정리 |
| **2. 응용 실습** | RSI 임계값·손절·거래비용·멀티종목 등 **직접 손대는 실습과제** |
| **3. 검증 방법론** | **In-Sample / Out-of-Sample 분리**, 과최적화 시연, **워크포워드 분석** |

> ### ⚠️ 왜 이 챕터가 제일 중요한가?
> 백테스팅에서 예쁜 우상향 곡선을 만드는 것은 **누구나 할 수 있습니다.**
> 파라미터(RSI 임계값 등)를 과거 데이터에 맞추면(curve-fitting) 수익률은 얼마든지 올라갑니다.
> 하지만 **그 전략이 미래에도 통할지**는 완전히 다른 문제이며, 이 간극을 메우는 표준 방법론이 **워크포워드 테스트**입니다.


---
## 0. 환경 설정

Colab에서 바로 실행되도록 데이터는 `FinanceDataReader`로 내려받습니다.
(원본 실습의 `005930.parquet` 파일이 없어도 동작합니다.)


In [ ]:
# Colab 환경 세팅
!pip install -q finance-datareader

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
# 그래프 축 라벨은 한글 폰트 깨짐을 피하기 위해 영문으로 표기합니다.


---
## 1. 데이터 준비

삼성전자(005930)를 사용하되, **200일선 워밍업 + 워크포워드 다구간 분할**을 위해 기간을 넉넉히(2015년~) 받습니다.


In [ ]:
# 삼성전자 일봉 데이터 (컬럼명을 소문자 'close'로 통일)
raw = fdr.DataReader('005930', '2015')
df = raw.rename(columns=str.lower)[['open', 'high', 'low', 'close', 'volume']].copy()

print(f"기간: {df.index[0].date()} ~ {df.index[-1].date()}  (총 {len(df)} 거래일)")
df.tail()


---
## 2. 지표 계산 & 전략 함수화 (Refactoring)

### 2-1. RSI(Relative Strength Index) 계산

RSI는 최근 상승폭과 하락폭의 상대적 크기를 0~100으로 나타내는 **모멘텀 지표**입니다.
- 보통 **30 이하는 과매도**, **70~80 이상은 과매수**로 봅니다.
- 아래는 표준(Wilder) 방식이며, `ewm`(지수가중이동평균)으로 상승/하락폭을 평활합니다.

> 📌 `diff`, `ewm` 모두 **과거 방향**만 참조하므로 look-ahead(미래참조)가 없습니다.


In [ ]:
def compute_rsi(close, period=14):
    """Wilder 방식 RSI. (0~100)"""
    delta = close.diff()
    gain = delta.clip(lower=0)          # 상승분
    loss = -delta.clip(upper=0)         # 하락분(양수화)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)


### 2-2. 백테스팅 함수 `run_backtest()`

원본처럼 로직을 셀마다 복붙하지 않고, **전략 파라미터를 인자로 받는 함수**로 만듭니다.

- **레짐**: `close > 200일 이동평균` 일 때만 신규 진입 허용
- **진입**: 현금 보유(flat) 상태에서 RSI가 `rsi_entry`(=45)를 **상향 돌파**하면 **가용현금 전액**으로 매수
- **청산**: RSI가 `rsi_exit`(=80) **이상**이 되면 전량 매도
- 매수/매도 모두 **슬리피지** 반영, 계산 단순화를 위해 **소수주(fractional share)** 허용

> 💡 원본은 "1주씩" 매수해 포트폴리오 대비 노출이 미미했습니다.
> 여기서는 **진입 시 전액 투자(all-in) → 청산 시 전액 현금화**로 바꿔
> Buy & Hold와 동일 선상에서 비교할 수 있게 합니다.


In [ ]:
def run_backtest(df, trend_window=200, rsi_period=14, rsi_entry=45, rsi_exit=80,
                 slippage=0.004, init_cash=1_000_000, start=None, end=None):
    """추세 필터 + RSI 진입/청산 전략 백테스트.

    매수: (flat) & (종가 > trend_window 이평) & (RSI가 rsi_entry를 상향 돌파)
    매도: (보유) & (RSI >= rsi_exit)

    Returns
    -------
    equity : pd.Series   # start~end 구간의 일별 총 포트폴리오 가치
    """
    data = df.copy()
    # 지표는 전체 구간에서 계산 (rolling/ewm은 과거만 참조 → look-ahead 없음)
    data['trend_ma'] = data['close'].rolling(trend_window).mean()
    data['rsi'] = compute_rsi(data['close'], rsi_period)
    data['rsi_prev'] = data['rsi'].shift(1)

    sim = data.loc[start:end]  # 시뮬레이션 구간만 슬라이스

    cash, shares = float(init_cash), 0.0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        ready = not (np.isnan(row['trend_ma']) or np.isnan(row['rsi']) or np.isnan(row['rsi_prev']))
        if ready:
            # 진입: 현금 상태 + 상승추세(200일선 위) + RSI가 rsi_entry(기본 45)를 상향 돌파
            crossed_up = (row['rsi_prev'] < rsi_entry) and (row['rsi'] >= rsi_entry)
            if shares == 0 and price > row['trend_ma'] and crossed_up:
                shares = cash / (price * (1 + slippage))   # 슬리피지 반영 매수
                cash = 0.0
            # 청산: RSI가 80 도달
            elif shares > 0 and row['rsi'] >= rsi_exit:
                cash = shares * price * (1 - slippage)      # 슬리피지 반영 매도
                shares = 0.0
        equity.append(cash + shares * price)

    return pd.Series(equity, index=sim.index, name='equity')


### 2-3. 성과지표 함수화

**총수익률 / CAGR / Sharpe / MDD** 를 한 함수로 묶습니다.


In [ ]:
def performance(equity, periods_per_year=250, rf=0.0):
    """equity curve로부터 핵심 성과지표를 계산해 dict로 반환."""
    equity = pd.Series(equity).reset_index(drop=True).astype(float)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    years = len(equity) / periods_per_year
    cagr = (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan

    daily = equity.pct_change().dropna()
    excess = daily - rf / periods_per_year
    sharpe = (excess.mean() / daily.std()) * np.sqrt(periods_per_year) if daily.std() > 0 else np.nan

    dd = equity / equity.cummax() - 1
    mdd = dd.min()

    return {'total_return': total_return, 'cagr': cagr, 'sharpe': sharpe, 'mdd': mdd}


def summary(name, equity):
    m = performance(equity)
    print(f"[{name}]")
    print(f"  총수익률 : {m['total_return']*100:7.2f}%")
    print(f"  CAGR     : {m['cagr']*100:7.2f}%")
    print(f"  Sharpe   : {m['sharpe']:7.2f}")
    print(f"  MDD      : {m['mdd']*100:7.2f}%")
    return m


### 2-4. 기본 파라미터로 전체 기간 백테스트 (200일선 + RSI 45/80)

In [ ]:
eq_base = run_backtest(df, trend_window=200, rsi_period=14,
                       rsi_entry=45, rsi_exit=80, slippage=0.004)

# 벤치마크: Buy & Hold
bh = df['close'] / df['close'].iloc[0] * 1_000_000

summary('Strategy (전체기간)', eq_base)
print('-' * 40)
summary('Buy & Hold', bh)

plt.plot(eq_base.index, eq_base.values, 'k', label='Strategy (Trend+RSI)')
plt.plot(bh.index, bh.values, 'r', alpha=0.7, label='Buy & Hold')
plt.title('Strategy vs Buy & Hold (Full Period)')
plt.ylabel('Portfolio Value'); plt.legend(); plt.show()


### 2-5. (참고) 매매 신호를 눈으로 확인

RSI와 진입/청산 지점을 가격 차트 위에 찍어 전략이 의도대로 동작하는지 확인합니다.


In [ ]:
# 지표 재계산(시각화용)
viz = df.copy()
viz['trend_ma'] = viz['close'].rolling(200).mean()
viz['rsi'] = compute_rsi(viz['close'], 14)
viz['rsi_prev'] = viz['rsi'].shift(1)

ENTRY, EXIT = 45, 80
buy_sig  = viz[(viz['close'] > viz['trend_ma']) &
               (viz['rsi_prev'] < ENTRY) & (viz['rsi'] >= ENTRY)]
sell_sig = viz[viz['rsi'] >= EXIT]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(viz.index, viz['close'], 'gray', lw=0.8, label='Close')
ax1.plot(viz.index, viz['trend_ma'], 'b', lw=1, label='MA200')
ax1.scatter(buy_sig.index, buy_sig['close'], marker='^', c='g', s=40, label='Buy signal')
ax1.scatter(sell_sig.index, sell_sig['close'], marker='v', c='r', s=15, label='Sell(RSI>=80)')
ax1.legend(); ax1.set_title('Price with MA200 & Signals')

ax2.plot(viz.index, viz['rsi'], 'purple', lw=0.8)
ax2.axhline(ENTRY, color='g', ls='--', lw=0.8); ax2.axhline(EXIT, color='r', ls='--', lw=0.8)
ax2.set_ylabel('RSI'); ax2.set_ylim(0, 100)
plt.show()


### 2-6. 🔍 진단: "왜 RSI 30 돌파 조건은 거의 안 걸릴까?"

처음 이 전략을 **RSI 30 상향 돌파**로 설계하면, 백테스트 곡선이 어느 시점(예: 강한 상승장 진입) 이후
**완전히 평평**해집니다 — 청산 후 **재진입 신호가 뜨지 않아** 현금만 들고 있게 되기 때문입니다.

원인은 RSI 계산 오류가 **아니라**, 진입 조건 두 개가 **서로 반대 국면**에서 발생한다는 데 있습니다.

- `RSI < 30` (깊은 과매도) → 보통 **급락 = 하락장** → 가격이 **200일선 아래** → 레짐 필터가 막음
- `price > 200일선` (상승장) → RSI가 30까지 **잘 안 내려감** (상승장에선 눌려도 40~50에서 반등)

아래 셀로 **직접 세어** 확인해 봅니다. (RSI 자체는 30 밑으로 잘 내려가지만, '200일선 위'와 겹치는 날이 거의 없음)


In [ ]:
# 진단: 조건별 발생일수와 진입신호 수
diag = df.copy()
diag['ma200']    = diag['close'].rolling(200).mean()
diag['rsi']      = compute_rsi(diag['close'], 14)
diag['rsi_prev'] = diag['rsi'].shift(1)
diag['above_ma'] = diag['close'] > diag['ma200']
diag['under30']  = diag['rsi'] < 30

both = int((diag['under30'] & diag['above_ma']).sum())
print(f"RSI<30 인 날            : {int(diag['under30'].sum()):4d} 일")
print(f"그 중 price>MA200 동시만족: {both:4d} 일   ← 실제로 진입 가능한 날")
print("→ 과매도(RSI<30)는 대개 하락장(price<MA200)에서 발생 → 두 조건이 배타적\n")

print("진입 임계값별 신호 수 (200일선 위 & 상향돌파):")
for lv in [30, 40, 45, 50]:
    cross = (diag['rsi_prev'] < lv) & (diag['rsi'] >= lv)
    n = int((diag['above_ma'] & cross).sum())
    tag = "  ← 현재 기본값" if lv == 45 else ("  ← 거의 거래 안 됨" if lv == 30 else "")
    print(f"  RSI {lv} 돌파 진입: {n:4d} 회{tag}")

# 연도별로도 확인 (강한 상승장일수록 RSI<30 이 사라짐)
diag['cross45'] = (diag['rsi_prev'] < 45) & (diag['rsi'] >= 45)
diag['entry45'] = diag['above_ma'] & diag['cross45']
print("\n연도별 진입신호(RSI45) / RSI<30 일수:")
yr = diag.assign(year=diag.index.year).groupby('year').agg(
    entry_45=('entry45', 'sum'), rsi_under30=('under30', 'sum'))
print(yr.to_string())


> **결론**
> - **RSI 계산은 정상**입니다(교과서 Wilder 방식과 워밍업 이후 소수점까지 일치).
> - 문제는 `RSI<30` + `200일선 위` 조합이 **거의 동시에 성립하지 않는 설계**였습니다.
> - 그래서 이 노트북의 기본 진입값을 **RSI 45 상향 돌파**로 두었습니다 — 상승 추세의 **얕은 눌림목**을 잡아
>   전 구간에 걸쳐 꾸준히 매매가 발생합니다.
> - (원래 "30 돌파"를 쓰고 싶다면 레짐 필터를 완화하거나, 진입을 "RSI가 30 아래로 갔다가 되돌아올 때"로
>   더 길게 관찰하는 등 **조건을 느슨하게** 해야 합니다. → 실습 3-1에서 실험해 보세요.)


---
## 3. 응용 실습과제 (기본기 → 응용)

함수가 생겼으니 **파라미터/로직을 바꿔가며 실험**할 수 있습니다.
아래 과제들을 직접 채워 보세요. (각 셀의 `# TODO`)

> 💡 "하나의 최고값"을 찾기보다, **결과가 왜/어떻게 달라지는지 해석**하는 것이 핵심입니다.


### 실습 3-1. RSI 임계값 민감도 분석

청산 기준 `rsi_exit` 를 65, 70, 75, 80, 85로 바꾸며 성과가 어떻게 변하는지 표로 정리하세요.
**"하나의 정답"이 아니라 "안정적으로 잘 되는 구간"** 이 있는지 관찰하는 것이 목적입니다.


In [ ]:
# 실습 3-1 예시 코드
rows = []
for ex in [65, 70, 75, 80, 85]:
    eq = run_backtest(df, rsi_exit=ex)
    rows.append({'rsi_exit': ex, **performance(eq)})

result_31 = pd.DataFrame(rows).set_index('rsi_exit')
print(result_31.round(3))

# TODO: rsi_entry(35,40,45,50)나 rsi_period(7,14,21)로도 같은 분석을 해보세요.
#       특히 rsi_entry=30 으로 두면 진입이 거의 없어 곡선이 평평해지는 것을 확인하세요.


### 실습 3-2. 손절(Stop-Loss) 추가

이 전략은 RSI 80에 도달할 때까지 기다립니다. 만약 진입 후 추세가 꺾여 RSI 80에 영영 못 가면
큰 손실로 이어질 수 있습니다. **진입가 대비 일정 % 이상 하락하면 즉시 매도**하는 손절을 추가하세요.


In [ ]:
def run_backtest_sl(df, trend_window=200, rsi_period=14, rsi_entry=45, rsi_exit=80,
                    stop_loss=0.07, slippage=0.004, init_cash=1_000_000,
                    start=None, end=None):
    data = df.copy()
    data['trend_ma'] = data['close'].rolling(trend_window).mean()
    data['rsi'] = compute_rsi(data['close'], rsi_period)
    data['rsi_prev'] = data['rsi'].shift(1)
    sim = data.loc[start:end]

    cash, shares, entry_price = float(init_cash), 0.0, 0.0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        ready = not (np.isnan(row['trend_ma']) or np.isnan(row['rsi']) or np.isnan(row['rsi_prev']))
        if ready:
            crossed_up = (row['rsi_prev'] < rsi_entry) and (row['rsi'] >= rsi_entry)
            if shares == 0 and price > row['trend_ma'] and crossed_up:
                shares = cash / (price * (1 + slippage)); cash = 0.0; entry_price = price
            # TODO: 손절 조건을 완성하세요.
            #   힌트) shares > 0 이고 price <= entry_price * (1 - stop_loss) 이면 전량 매도
            elif shares > 0 and row['rsi'] >= rsi_exit:
                cash = shares * price * (1 - slippage); shares = 0.0
        equity.append(cash + shares * price)
    return pd.Series(equity, index=sim.index)

# TODO: stop_loss=0.07 로 실행 후 원본과 MDD/CAGR 비교


### 실습 3-3. 거래비용 민감도

`slippage` 를 0%, 0.2%, 0.4%, 1.0%로 바꾸며 **CAGR이 얼마나 깎이는지** 확인하세요.
> 실전에서는 슬리피지 + 수수료 + 세금(매도 시 거래세)까지 반영해야 합니다.
> **거래가 잦은 전략일수록 비용에 취약**하다는 점을 수치로 체감해 보세요.


In [ ]:
# 실습 3-3 예시
for s in [0.0, 0.002, 0.004, 0.01]:
    m = performance(run_backtest(df, slippage=s))
    print(f"slippage {s*100:.1f}% -> CAGR {m['cagr']*100:6.2f}% | Sharpe {m['sharpe']:.2f}")


### 실습 3-4. (도전) 멀티 종목으로 확장

`fdr.DataReader` 로 종목 여러 개(예: 000660 SK하이닉스, 005380 현대차)를 받아
각각 백테스트한 뒤 **동일가중 합산 포트폴리오**의 성과를 구해 보세요.
단일 종목 대비 MDD가 줄어드는지(분산 효과) 확인하는 것이 포인트입니다.


In [ ]:
# 실습 3-4 스켈레톤
tickers = ['005930', '000660', '005380']
equities = {}
for t in tickers:
    d = fdr.DataReader(t, '2015').rename(columns=str.lower)
    equities[t] = run_backtest(d[['close']], init_cash=1_000_000)

# TODO: 세 equity curve를 날짜 기준 정렬 후 동일가중 합산하여 포트폴리오 성과 계산
# port = pd.concat(equities, axis=1).ffill().sum(axis=1)
# summary('3-Stock Portfolio', port)


---
## 4. 핵심 개념: In-Sample / Out-of-Sample 과 과최적화

### 4-1. 왜 "전체 기간 최적화"는 위험한가?

전략을 만들 때 흔히 이렇게 합니다.

> "여러 파라미터 조합(RSI 진입/청산값 등)을 **전체 기간**에 돌려보고, 가장 수익률 좋은 걸 고른다."

이건 **시험 문제를 미리 보고 답을 외우는 것(curve-fitting)** 과 같습니다.
과거에 가장 잘 맞은 파라미터가 미래에도 최적이라는 보장이 없으며,
오히려 **과거의 노이즈(우연)에 맞춰졌을** 가능성이 큽니다. 이것이 **과최적화(Overfitting)** 입니다.

### 4-2. 해결의 출발점: 데이터를 나눈다

| 구분 | 이름 | 역할 |
|------|------|------|
| **IS** | In-Sample (학습/최적화 구간) | 파라미터를 **찾는** 데 사용 |
| **OOS** | Out-of-Sample (검증 구간) | 찾은 파라미터를 **처음 보는 데이터로 검증** |

**IS에서 고른 파라미터를, IS 성과가 아니라 OOS 성과로 평가**하는 것이 핵심입니다.
OOS에서 성과가 무너지면 → 그 전략은 **과최적화된 것**입니다.


### 4-3. 시연: IS에서 '최적' RSI 파라미터를 찾고 OOS에서 검증

그리드 서치 함수를 만들고, 데이터를 앞(IS)/뒤(OOS)로 나눠 봅니다.
(추세 필터 200일선은 전략의 정체성이라 고정하고, **RSI 파라미터를 탐색**합니다.)


In [ ]:
def grid_search(df, period_list, entry_list, exit_list, start, end,
                trend_window=200, metric='sharpe'):
    """[start, end] 구간에서 RSI 파라미터 조합을 전수 탐색, metric 기준 내림차순 정렬."""
    rows = []
    for p, en, ex in itertools.product(period_list, entry_list, exit_list):
        eq = run_backtest(df, trend_window=trend_window, rsi_period=p,
                          rsi_entry=en, rsi_exit=ex, start=start, end=end)
        rows.append({'rsi_p': p, 'entry': en, 'exit': ex, **performance(eq)})
    return pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)


# 탐색할 파라미터 그리드
PERIOD_LIST = [7, 14, 21]
ENTRY_LIST  = [40, 45, 50]
EXIT_LIST   = [65, 70, 75, 80]

# IS / OOS 분리 (앞 70% 학습, 뒤 30% 검증)
split = df.index[int(len(df) * 0.7)]
print(f"IS : {df.index[0].date()} ~ {split.date()}")
print(f"OOS: {split.date()} ~ {df.index[-1].date()}")


In [ ]:
# 1) IS 구간에서 '최적' 파라미터 탐색
is_result = grid_search(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST,
                        start=None, end=split, metric='sharpe')
print('=== IS 상위 5개 조합 ===')
print(is_result.head().round(3))

best = is_result.iloc[0]
print(f"\n선택된 최적 파라미터: rsi_period={int(best.rsi_p)}, "
      f"entry={int(best.entry)}, exit={int(best['exit'])}")


In [ ]:
# 2) 같은 파라미터를 IS와 OOS에 각각 적용해 성과 비교
p = dict(rsi_period=int(best.rsi_p), rsi_entry=int(best.entry), rsi_exit=int(best['exit']))

eq_is  = run_backtest(df, **p, start=None,  end=split)
eq_oos = run_backtest(df, **p, start=split, end=None)

print('>>> IS(학습 구간) 성과 — 당연히 좋아 보임')
summary('IS', eq_is)
print('-' * 40)
print('>>> OOS(검증 구간) 성과 — 진짜 실력')
summary('OOS', eq_oos)


> **관찰 포인트**
> IS에서 Sharpe가 가장 높았던 파라미터를 OOS에 적용하면 성과(Sharpe/CAGR)가 **눈에 띄게 하락**하는 경우가 많습니다.
> IS 성과와 OOS 성과의 **격차가 클수록 과최적화가 심한 것**입니다.


---
## 5. 워크포워드 분석 (Walk-Forward Analysis)

### 5-1. 한 번의 IS/OOS 분리로는 부족하다

4장의 방법은 좋지만, **딱 한 번** 나눈 것이라 운(luck)의 영향이 큽니다.
하필 OOS 구간이 대세 상승장이면 아무 전략이나 잘 나오고, 하락장이면 다 나쁘게 나옵니다.

**워크포워드**는 이 IS→OOS 검증을 **시간 축을 따라 여러 번 반복**합니다.

```
[  IS #1  ][OOS#1]
        [  IS #2  ][OOS#2]
                [  IS #3  ][OOS#3]
                        [  IS #4  ][OOS#4] ...
```

각 단계에서:
1. **IS 구간**에서 최적 RSI 파라미터를 찾고
2. 그 파라미터를 **바로 뒤 OOS 구간**에 적용해 성과를 기록한 뒤
3. 창(window)을 OOS 길이만큼 앞으로 밀어 반복

그리고 **모든 OOS 구간을 이어붙인 곡선** — 이것이 실전에 가장 가까운 성과 추정치입니다.
(주기적으로 파라미터를 재조정하며 굴리는 실제 운용을 그대로 흉내 낸 것)


In [ ]:
def walk_forward(df, period_list, entry_list, exit_list,
                 is_len=500, oos_len=125, trend_window=200,
                 metric='sharpe', init_cash=1_000_000):
    """롤링 워크포워드 분석.

    is_len  : In-Sample 길이(거래일). 500 ≈ 2년
    oos_len : Out-of-Sample 길이(거래일). 125 ≈ 6개월
    Returns
    -------
    wf_equity : pd.Series      # 이어붙인 OOS equity curve
    log       : pd.DataFrame   # 각 fold에서 선택된 파라미터/성과
    """
    idx = df.index
    n = len(df)
    segments, log = [], []
    i = 0
    cash = init_cash
    while i + is_len + oos_len <= n:
        is_start,  is_end  = idx[i],           idx[i + is_len - 1]
        oos_start, oos_end = idx[i + is_len],  idx[i + is_len + oos_len - 1]

        # 1) IS에서 최적 파라미터
        res = grid_search(df, period_list, entry_list, exit_list,
                          is_start, is_end, trend_window, metric)
        b = res.iloc[0]

        # 2) OOS에 적용 (자본은 이전 fold에서 이어받음)
        eq = run_backtest(df, trend_window=trend_window, rsi_period=int(b.rsi_p),
                          rsi_entry=int(b.entry), rsi_exit=int(b['exit']),
                          init_cash=cash, start=oos_start, end=oos_end)
        cash = eq.iloc[-1]
        segments.append(eq)

        m = performance(eq)
        log.append({'oos_start': oos_start.date(), 'oos_end': oos_end.date(),
                    'rsi_p': int(b.rsi_p), 'entry': int(b.entry), 'exit': int(b['exit']),
                    'oos_cagr': m['cagr'], 'oos_sharpe': m['sharpe'], 'oos_mdd': m['mdd']})
        i += oos_len

    wf_equity = pd.concat(segments)
    return wf_equity, pd.DataFrame(log)


In [ ]:
wf_equity, wf_log = walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST,
                                 is_len=500, oos_len=125, metric='sharpe')

print('=== 각 OOS 구간에서 선택된 파라미터와 성과 ===')
print(wf_log.round(3).to_string(index=False))
print()
summary('Walk-Forward (이어붙인 OOS)', wf_equity)


### 5-2. 결정적 비교: 전체최적화(과최적화) vs 워크포워드(정직한 추정)

같은 파라미터 그리드로,
- **(A) 전체 기간 최적화**: 전체 데이터에서 가장 좋은 파라미터 하나를 골라 전체에 적용 → *실전에서는 불가능한 "미래를 아는" 성과*
- **(B) 워크포워드 OOS**: 매 시점 과거만 보고 파라미터를 정한 성과 → *실전에서 실제로 얻었을 성과*

두 곡선의 격차가 바로 **"백테스트 낙관 편향(optimism bias)"** 의 크기입니다.


In [ ]:
# (A) 전체 기간 최적화 — 미래를 훔쳐본 결과
full_best = grid_search(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST,
                        start=None, end=None, metric='sharpe').iloc[0]
eq_full = run_backtest(df, rsi_period=int(full_best.rsi_p),
                       rsi_entry=int(full_best.entry), rsi_exit=int(full_best['exit']))

# 비교 구간을 워크포워드 OOS 시작점에 맞춤
eq_full_aligned = eq_full.loc[wf_equity.index[0]:]
eq_full_aligned = eq_full_aligned / eq_full_aligned.iloc[0] * 1_000_000
wf_norm = wf_equity / wf_equity.iloc[0] * 1_000_000

print('>>> (A) 전체최적화 = 과최적화된 "이상적" 성과')
summary('Full-Sample Optimized', eq_full_aligned)
print('-' * 40)
print('>>> (B) 워크포워드 = 정직한 성과 추정')
summary('Walk-Forward OOS', wf_norm)

plt.plot(eq_full_aligned.index, eq_full_aligned.values, 'r', label='(A) Full-Sample Optimized (overfit)')
plt.plot(wf_norm.index, wf_norm.values, 'k', label='(B) Walk-Forward OOS (honest)')
plt.title('Overfitted Backtest vs Walk-Forward'); plt.ylabel('Portfolio Value')
plt.legend(); plt.show()


> **해석**
> (A) 빨간 곡선(전체최적화)이 (B) 검은 곡선(워크포워드)보다 훨씬 좋아 보인다면,
> 그 차이는 **실력이 아니라 미래를 훔쳐본 대가**입니다.
> 실전에 배포하면 우리가 실제로 얻는 것은 (B)에 가깝습니다.
>
> **워크포워드 성과가 벤치마크(Buy&Hold) 대비, 그리고 거래비용 반영 후에도 살아남아야**
> 비로소 "쓸 만한 전략"이라고 말할 수 있습니다.


### 5-3. 파라미터 안정성(Parameter Stability) 관찰

`wf_log` 를 보면 fold마다 선택된 RSI 파라미터가 **얼마나 흔들리는지** 알 수 있습니다.
- fold마다 파라미터가 **크게 요동친다** → 전략이 데이터에 과민(불안정), 신뢰도 낮음
- fold가 달라도 **비슷한 파라미터가 반복 선택** → 견고한(robust) 신호일 가능성

좋은 전략은 "특정 마법의 숫자"가 아니라 **넓은 파라미터 구간에서 고르게 잘 되는** 전략입니다.


In [ ]:
# fold별 선택 파라미터 변동 시각화
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, col in zip(axes, ['rsi_p', 'entry', 'exit']):
    ax.plot(range(len(wf_log)), wf_log[col], 'o-')
    ax.set_ylabel(col)
axes[0].set_title('Selected RSI Parameters per Walk-Forward Fold')
axes[-1].set_xlabel('Fold #')
plt.show()


---
## 6. 심화 실습과제

1. **Anchored(고정 시작점) 워크포워드 구현**
   - 위 `walk_forward` 는 IS 창이 앞으로 밀리는 **Rolling** 방식입니다.
   - IS 시작점을 고정하고 창을 계속 **늘려가는** Anchored 방식으로 바꿔, 두 방식의 OOS 성과를 비교하세요.

2. **`is_len` / `oos_len` 민감도**
   - `is_len` 을 250/500/750, `oos_len` 을 63/125/250으로 바꿔가며 워크포워드를 돌리고,
     OOS 성과가 이 설정에 얼마나 민감한지 표로 정리하세요.
   - *한 설정에서만 잘 나온다면 그 자체가 위험 신호입니다.*

3. **파라미터 안정성 히트맵**
   - 특정 IS 구간에서 (entry × exit) 조합별 Sharpe를 2D 히트맵으로 그려,
     "최적점"이 **뾰족한 봉우리인지 넓은 고원인지** 확인하세요. (고원이 더 신뢰할 만함)

4. **거래비용을 반영한 워크포워드**
   - `run_backtest` 의 `slippage` 를 현실적으로 높인 뒤(예: 0.5%),
     워크포워드 OOS 성과가 벤치마크를 여전히 이기는지 검증하세요.

5. **레짐 필터 효과 분리**
   - 200일선 필터를 **끄고**(`trend_window=1`) 동일한 RSI 전략을 워크포워드로 검증한 뒤,
     필터가 있을 때와 성과(특히 MDD)를 비교하세요. *추세 필터가 실제로 손실을 줄이는지* 확인합니다.


---
## 7. 핵심 요약

- **백테스팅 코드는 함수화**해야 파라미터 실험·검증이 가능하다.
- 예쁜 전체기간 수익곡선은 **과최적화의 산물**일 수 있다 — 반드시 의심하라.
- **In-Sample / Out-of-Sample 분리**: 파라미터는 IS에서 찾고, 평가는 OOS로 한다.
- **워크포워드 분석**: IS→OOS 검증을 시간축으로 반복해 이어붙인 OOS 곡선이 **실전에 가장 가까운 성과**다.
- 좋은 전략의 조건: **① OOS에서 살아남고 ② 거래비용 후에도 벤치마크를 이기며 ③ 파라미터가 안정적**이다.

> "In-Sample에서 잘 되는 전략은 널렸다. **Out-of-Sample에서 살아남는 전략만이 돈을 번다.**"
